In [42]:
import numpy as np
import pandas as pd

In [43]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [44]:
df = pd.read_csv('covid_toy.csv')

In [45]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [46]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [47]:
from sklearn.model_selection import train_test_split as tts
X_train, X_test, y_train, y_test = tts(df.drop(columns=['has_covid']),df['has_covid'], test_size=0.2)

In [48]:
X_train

,age,gender,fever,cough,city
16,69,Female,103.0,Mild,Kolkata
27,33,Female,102.0,Strong,Delhi
8,19,Female,100.0,Strong,Bangalore
95,12,Female,104.0,Mild,Bangalore
83,17,Female,104.0,Mild,Kolkata
...,...,...,...,...,...
59,6,Female,104.0,Mild,Kolkata
68,54,Female,104.0,Strong,Kolkata
86,25,Male,104.0,Mild,Bangalore
81,65,Male,99.0,Mild,Delhi


Manually doing column transformer

In [49]:
#In default SimpleImputer replaces null data with mean(default)
si = SimpleImputer(strategy='mean')
X_train_fever = si.fit_transform(X_train[['fever']])

X_test_fever = si.fit_transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

In [50]:
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [51]:
ohe = OneHotEncoder(drop='first', sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [52]:
#Extracting Age
X_train_age = X_train.drop(columns=['gender', 'fever', 'cough', 'city']).values

X_test_age = X_test.drop(columns=['gender', 'fever', 'cough', 'city']).values
X_train_age.shape


(80, 1)

In [53]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape


(80, 7)

Now with Column Transformer

In [54]:
from sklearn.compose import ColumnTransformer

In [55]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [56]:
transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(),['fever']),
    ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]),['cough']),
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first'),['gender','city'])
], remainder='passthrough')

In [57]:
transformer.fit_transform(X_train).shape

(80, 7)

In [58]:
transformer.fit_transform(X_test).shape

(20, 7)